# 03 · Inferência

Gera as previsões da fase de grupos de 2026.

### Contrato
**Entradas:** `models/model.joblib`, `data/processed/features.parquet` (linhas `split=="predict"`: os 72 jogos de 2026 com as features já computadas pelo `01`), `data/raw/matches-schedule.csv`

**Saída (quando implementar):** `data/results/predictions_submission.csv`

> As features dos 72 jogos vêm prontas do `01` (mesma rotina de treino, sem *train/serve skew*). O `03` apenas carrega `split=="predict"`, aplica o modelo e escreve a submission.

In [1]:
# Setup: resolve a raiz do repositório e os caminhos de dados.
from pathlib import Path

def find_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for cand in [base, *base.parents]:
        if (cand / "data" / "raw").is_dir() and (cand / "pyproject.toml").is_file():
            return cand
    raise RuntimeError("Raiz do repositório não encontrada a partir de " + str(base))

ROOT = find_root()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
MODELS = ROOT / "models"
print("ROOT:", ROOT)
RESULTS.mkdir(parents=True, exist_ok=True)

ROOT: /Users/franklaercio/Workspace/paul-the-octopus


In [2]:
import pandas as pd

# Calendário cru (72 partidas) — referência de match/date.
calendario = pd.read_csv(RAW / 'matches-schedule.csv')

# Features dos 72 jogos de 2026, prontas do 01 (mesma rotina de treino).
features = pd.read_parquet(PROCESSED / 'features.parquet')
predict = features[features['split'] == 'predict'].reset_index(drop=True)
assert len(predict) == 72, f'esperado 72 jogos predict, obtido {len(predict)}'
predict.shape

(72, 31)

## Gere as previsões
Carregue o modelo do `02`, aplique ao calendário e escreva a submission no formato que você definir.

A submissão mantém o contrato 1X2 da skill (`match, home, away, p_home, p_draw, p_away`).
Se o bundle trouxer o **componente de gols companheiro** (chave `goals`), anexam-se
`xg_home`, `xg_away` (gols esperados λ, 1 casa) e `placar_provavel` — o placar é o
argmax da matriz de placares **restrito à região do 1X2 previsto**, então nunca
contradiz a coluna 1X2. As probabilidades 1X2 não mudam.

In [ ]:
import joblib
import numpy as np

# Carrega o bundle do 02 (pipeline fitted + meta autossuficiente).
bundle = joblib.load(MODELS / 'model.joblib')
pipe = bundle['pipeline']
meta = bundle['meta']
feature_names = meta['feature_names']   # ordem exata de entrada
classes = meta['classes']               # ['home','draw','away'] (ordem do RPS)
shrink_alpha = float(meta.get('shrink_alpha', 0.0))   # shrinkage para a taxa-base (0 = cru)
base_rate = np.asarray(meta.get('base_rate', [0.0, 0.0, 0.0]), dtype=float)
ensemble_weight = float(meta.get('ensemble_weight', 1.0))  # peso do clf no ensemble (1 = só clf)
print('modelo:', meta['model_name'], '| sklearn', meta['sklearn_version'],
      '| shrink_alpha:', shrink_alpha, '| ensemble_weight:', ensemble_weight)

# Probabilidades 1X2 do CLASSIFICADOR. classes_ do sklearn é alfabético (away,draw,
# home); mapeamos POR NOME para a ordem do RPS (home,draw,away) — guardrail R1. A
# combinação com o componente de gols (ensemble) e o shrinkage vêm na célula seguinte:
# o ensemble depende do 1X2 implícito de Dixon-Coles, que precisa do lambda dos gols.
proba_clf = pipe.predict_proba(predict[feature_names])
col_of = {c: list(pipe.classes_).index(c) for c in classes}
proba_clf = np.column_stack([proba_clf[:, col_of['home']],
                             proba_clf[:, col_of['draw']],
                             proba_clf[:, col_of['away']]])

In [ ]:
# Probabilidades 1X2 FINAIS = ENSEMBLE (classificador + 1X2 implícito Dixon-Coles) -> shrinkage.
# ensemble: P = w*clf + (1-w)*DC; o shrinkage para a taxa-base é aplicado por cima. As
# funções vêm da FONTE ÚNICA em scripts.run_pipeline (as MESMAS do 02/validate). Se o
# bundle não tiver 'goals' OU ensemble_weight==1, cai no classificador puro (retrocompat).
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.run_pipeline import (  # noqa: E402
    apply_shrinkage,
    ensemble_proba,
    implied_1x2_from_goals,
    predict_goals,
    scores_from_lambdas,
)

has_goals = bool(meta.get("has_goals")) and ("goals" in bundle)
if has_goals and ensemble_weight < 1.0:
    proba_dc = implied_1x2_from_goals(bundle["goals"], predict)
    proba_1x2 = ensemble_proba(proba_clf, proba_dc, ensemble_weight)
    print("ensemble ativo | w_clf:", ensemble_weight,
          "| p_draw médio: clf", round(proba_clf[:, 1].mean(), 4),
          "-> ensemble", round(proba_1x2[:, 1].mean(), 4))
else:
    proba_1x2 = proba_clf
    print("sem ensemble (bundle sem gols ou ensemble_weight=1) — classificador puro.")

# Shrinkage para a taxa-base (com alpha=0 é inócuo) e renormalização defensiva.
proba = apply_shrinkage(proba_1x2, base_rate, shrink_alpha)
proba = proba / proba.sum(axis=1, keepdims=True)
p_home, p_draw, p_away = proba[:, 0], proba[:, 1], proba[:, 2]
assert np.allclose(p_home + p_draw + p_away, 1.0)

# Placar companheiro: xg (λ) e placar_provavel restrito à região do 1X2 FINAL previsto
# (home -> h>a, draw -> h=a, away -> h<a). O placar nunca contradiz a coluna 1X2.
if has_goals:
    goals_bundle = bundle["goals"]
    rho = float(goals_bundle["rho"])
    lam_home, lam_away = predict_goals(goals_bundle, predict)
    outcomes_pred = [classes[i] for i in np.argmax(proba, axis=1)]
    xg, placar_provavel = scores_from_lambdas(lam_home, lam_away, rho, outcomes_pred)
    xg_home, xg_away = xg[:, 0], xg[:, 1]
    print("componente de gols ativo | rho:", round(rho, 4),
          "| xg_home[:3]:", xg_home[:3].tolist(),
          "| placar[:3]:", placar_provavel[:3])
else:
    print("bundle sem componente de gols — submissão fica só com 1X2 (retrocompatível).")

In [5]:
# Monta a submission no contrato da skill: match, home, away, p_home, p_draw, p_away.
# 'match' vem de match_no; home/away usam os nomes de exibição do calendário (a skill
# casa por 'match' e canonicaliza os nomes ao pontuar). As colunas 1X2 do contrato
# seguem IDÊNTICAS; xg_home/xg_away/placar_provavel são acréscimos (companheiro).
submission = predict[['match_no']].copy()
submission['match'] = submission['match_no'].astype(int)
disp = calendario[['match', 'home', 'away']].copy()
disp['match'] = disp['match'].astype(int)
submission = submission.merge(disp, on='match', how='left')
submission['p_home'] = p_home
submission['p_draw'] = p_draw
submission['p_away'] = p_away
cols = ['match', 'home', 'away', 'p_home', 'p_draw', 'p_away']

# Colunas de placar do companheiro (só quando há componente de gols): xg em 1 casa e
# placar_provavel consistente com o 1X2. Anexadas DEPOIS das 1X2 (contrato 1X2 intacto).
if has_goals:
    submission['xg_home'] = np.round(xg_home, 1)
    submission['xg_away'] = np.round(xg_away, 1)
    submission['placar_provavel'] = placar_provavel
    cols += ['xg_home', 'xg_away', 'placar_provavel']

submission = submission[cols].sort_values('match')

assert len(submission) == 72
assert submission[['home', 'away']].notna().all().all()
if has_goals:
    # invariante: o placar nunca contradiz o 1X2 previsto (home h>a, draw h=a, away h<a).
    _pred = submission[['p_home', 'p_draw', 'p_away']].to_numpy().argmax(axis=1)
    _ha = submission['placar_provavel'].str.split('-', expand=True).astype(int).to_numpy()
    _exp = np.where(_ha[:, 0] > _ha[:, 1], 0, np.where(_ha[:, 0] < _ha[:, 1], 2, 1))
    assert (_pred == _exp).all(), 'placar_provavel inconsistente com o 1X2'
submission.to_csv(RESULTS / 'predictions_submission.csv', index=False)
print('submission salva:', RESULTS / 'predictions_submission.csv')
submission.head()

submission salva: /Users/franklaercio/Workspace/paul-the-octopus/data/results/predictions_submission.csv


,match,home,away,p_home,p_draw,p_away,xg_home,xg_away,placar_provavel
0,1,Mexico,South Africa,0.740060,0.188599,0.071341,2.1,0.6,2-0
1,2,South Korea,Czech Republic,0.340861,0.282053,0.377086,1.2,1.3,0-1
2,3,Canada,Bosnia and Herzegovina,0.660886,0.210070,0.129044,1.8,0.7,1-0
3,4,USA,Paraguay,0.418340,0.264981,0.316679,1.3,1.3,1-0
6,5,Qatar,Switzerland,0.063930,0.159894,0.776176,0.5,2.4,0-2
